# Demos: 7 Combining Data

Each cell below is one demo from the slides; the heading matches the pill on the slide. Run the first cell, then run the demos in any order with Shift+Enter. The same code is in the `.py` files in this folder.

In [ ]:
# Run this cell first: the demos read their data files from the working folder.
import os
from pathlib import Path

COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir())
os.chdir(COURSE / ".")
print("Working folder:", Path.cwd())

## demo01 · Concat months

In [ ]:
from pathlib import Path

import pandas as pd

DATA = COURSE / "datasets"

months = ["jan", "feb", "mar"]
frames = [pd.read_csv(DATA / f"orders-{month}.csv", dtype={"customer_id": str}) for month in months]
orders = pd.concat(
    [frame.assign(source=month) for month, frame in zip(months, frames)], ignore_index=True
)
print(orders[["source", "order_id", "customer_id", "units"]].to_string(index=False))

indexed = pd.concat(frames, keys=months, names=["source", "row"])
print(indexed.index.tolist())
print("Rows:", len(orders), "Units:", orders["units"].sum())

## demo02 · Column alignment

In [ ]:
import pandas as pd

jan = pd.DataFrame({"id": ["a"], "units": [10]})
feb = pd.DataFrame({"units": [20], "id": ["b"], "channel": ["store"]})

combined = pd.concat([jan, feb], ignore_index=True)
print(combined.to_string(index=False))
shared = pd.concat([jan, feb], join="inner", ignore_index=True)
print("Shared columns:", shared.columns.tolist())

## demo03 · Merge and audit

In [ ]:
import pandas as pd

small_orders = pd.DataFrame({"customer_id": ["001", "002", "004"], "units": [10, 20, 30]})
small_customers = pd.DataFrame(
    {"customer_id": ["001", "002", "003"], "region": ["North", "South", "West"]}
)

for how in ["inner", "left", "outer"]:
    result = small_orders.merge(
        small_customers, on="customer_id", how=how, validate="one_to_one", indicator=True
    )
    print(how, "rows:", len(result), "units:", result["units"].sum())

audit = small_orders.merge(
    small_customers, on="customer_id", how="outer", validate="one_to_one", indicator=True
)
print(audit[["customer_id", "_merge"]].to_string(index=False))
unmatched = audit.loc[audit["_merge"] != "both"]
print("Unmatched keys:", unmatched["customer_id"].tolist())

left = small_orders.assign(status="ordered")
right = small_customers.rename(columns={"customer_id": "id"}).assign(status="active")
named = left.merge(
    right,
    left_on="customer_id",
    right_on="id",
    how="left",
    suffixes=("_order", "_customer"),
    validate="one_to_one",
)
print(named[["customer_id", "status_order", "status_customer"]].to_string(index=False))

## demo04 · Duplicate keys

In [ ]:
from pathlib import Path

import pandas as pd

DATA = COURSE / "datasets"
frames = [
    pd.read_csv(DATA / f"orders-{month}.csv", dtype={"customer_id": str})
    for month in ["jan", "feb", "mar"]
]
orders = pd.concat(frames, ignore_index=True)
customers = pd.read_csv(DATA / "orders-customers.csv", dtype={"customer_id": str})

try:
    orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
except pd.errors.MergeError:
    print("Rejected: customer lookup keys are not unique")
duplicates = customers.loc[customers.duplicated("customer_id", keep=False)]
print(duplicates.to_string(index=False))

bad = orders.merge(customers, on="customer_id", how="left")
print(bad[["order_id", "customer_id", "units"]].to_string(index=False))
print("Before:", len(orders), orders["units"].sum())
print("After:", len(bad), bad["units"].sum())

lookup = customers.drop_duplicates()  # documented exact repeated export
checked = orders.merge(lookup, on="customer_id", how="left", validate="many_to_one", indicator=True)
assert len(checked) == len(orders)
assert checked["units"].sum() == orders["units"].sum()
missing = checked.loc[checked["_merge"] == "left_only", "customer_id"]
print("Rows:", len(checked), "Units:", checked["units"].sum())
print("Missing lookup:", missing.tolist())

## demo05 · Key policies

In [ ]:
from pathlib import Path

import pandas as pd

DATA = COURSE / "datasets"

left = pd.DataFrame({"id": ["001", None], "units": [10, 20]})
right = pd.DataFrame({"id": ["001", None], "region": ["North", "Unknown"]})
matched = left.merge(right, on="id", validate="one_to_one", indicator=True)
print(matched.to_string(index=False))

inferred = pd.read_csv(DATA / "orders-jan.csv")
typed = pd.read_csv(DATA / "orders-jan.csv", dtype={"customer_id": str})
print("Inferred IDs:", inferred["customer_id"].tolist())
print("Text IDs:", typed["customer_id"].tolist())
try:
    inferred.merge(typed, on="customer_id")
except ValueError:
    print("Rejected: integer and string merge keys")

## demo06 · Index join

In [ ]:
import pandas as pd

small_orders = pd.DataFrame({"customer_id": ["001", "002", "004"], "units": [10, 20, 30]})
small_customers = pd.DataFrame(
    {"customer_id": ["001", "002", "003"], "region": ["North", "South", "West"]}
)

indexed_orders = small_orders.set_index("customer_id")
indexed_customers = small_customers.set_index("customer_id")
joined = indexed_orders.join(indexed_customers, how="left", validate="one_to_one")
print(joined)

## demo07 · Tidy reshape

In [ ]:
from pathlib import Path

import pandas as pd

DATA = COURSE / "datasets"
wide = pd.read_csv(DATA / "orders-wide.csv", dtype={"customer_id": str})

long = wide.melt(
    id_vars="customer_id", value_vars=["Jan", "Feb"], var_name="month", value_name="units"
)
print(long.to_string(index=False))
print("Measurements:", len(long), "Units:", long["units"].sum())

restored = long.pivot(index="customer_id", columns="month", values="units")
print(restored)

repeated = pd.concat([long, long.iloc[[0]]], ignore_index=True)
try:
    repeated.pivot(index="customer_id", columns="month", values="units")
except ValueError:
    print("Pivot rejected duplicate customer/month pairs")
totals = repeated.pivot_table(index="customer_id", columns="month", values="units", aggfunc="sum")
print(totals)